# Model 2 + 신규 터빈 LSTM Fast v3 — 연도전진 검증판

이 노트북은 기존 전처리 파일을 읽고 모델만 학습합니다. 옛 LSTM 생성 코드와 2-stage 잔차 모델은 포함하지 않습니다.

검증 순서:

1. Group 1·2는 2022년으로 학습하여 2023년에서 모든 혼합비율·하한·MoE·후처리를 선택합니다.
2. Group 3은 데이터가 2023년부터 있으므로 2023년 상반기로 학습하고 하반기에서 선택합니다.
3. 선택한 값을 고정한 뒤 과거 전체로 재학습하여 2024년 Q3·Q4를 독립 감사합니다.
4. Q3, Q4, 전체 점수가 모두 최소 기준 이상 개선된 그룹만 최종 구조를 사용합니다.
5. 검증 통과 후에만 2024년을 포함해 최종 재학습합니다.

포함 기능: 신규 터빈 LSTM 피처, LightGBM·CatBoost 앙상블, Active Classifier, 0.16~0.18 CF 하한 Anchor,
Group 2 Gaussian MoE, Group 3 고출력 확률 Gate·Quantile 전문가, Group 1 그룹 참조 후처리.

## 1. 환경과 경로

In [1]:
from pathlib import Path
import gc, json, os, random, time, warnings

import joblib
import numpy as np
import pandas as pd
import lightgbm as lgb
from catboost import CatBoostRegressor
from sklearn.metrics import roc_auc_score
from IPython.display import display

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ROOT = Path(r"C:\jupyter project")
MODE = "fast"
PREPROCESSED_DIR = ROOT / "wind_output" / "lstm_turbine_feature_fast_v2" / "preprocessed"
OUTPUT_DIR = ROOT / "wind_output" / "model_2_with_lstm_fast_v3_yearforward"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "checkpoints").mkdir(exist_ok=True)

TIME_COL = "forecast_kst_dtm"
GROUPS = (1, 2, 3)
CAPACITY = {1: 21600.0, 2: 21600.0, 3: 21000.0}
TARGET = {g: f"kpx_group_{g}" for g in GROUPS}
ACTIVE_CF = 0.10
PRED_MAX_CF = 1.05
MIN_Q_GAIN = 0.0015
MIN_FULL_GAIN = 0.0020
FINAL_ITERATION_MULTIPLIER = 1.10

if MODE == "fast":
    MAX_FEATURES = 110
    LGB_TREES = 900
    CAT_TREES = 850
    BLEND_TRIALS = 1300
    MOE_TRIALS = 700
    ACTIVE_CLASSIFIER_TREES = 650
    HIGH_CLASSIFIER_TREES = 500
    G2_EXPERT_TREES = 650
else:
    MAX_FEATURES = 180
    LGB_TREES = 2200
    CAT_TREES = 1800
    BLEND_TRIALS = 5000
    MOE_TRIALS = 3500
    ACTIVE_CLASSIFIER_TREES = 1500
    HIGH_CLASSIFIER_TREES = 1200
    G2_EXPERT_TREES = 1500

required = [PREPROCESSED_DIR / f"{split}_group{g}_preprocessed.csv"
            for split in ("train", "test") for g in GROUPS]
missing = [str(p) for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError("신규 터빈 LSTM 전처리 파일이 없습니다:\n- " + "\n- ".join(missing))

print("모드:", MODE)
print("입력:", PREPROCESSED_DIR)
print("출력:", OUTPUT_DIR)
print("검증: 선택연도에서 설정 고정 → 2024 Q3/Q4 독립 감사")

모드: fast
입력: C:\jupyter project\wind_output\lstm_turbine_feature_fast_v2\preprocessed
출력: C:\jupyter project\wind_output\model_2_with_lstm_fast_v3_yearforward
검증: 선택연도에서 설정 고정 → 2024 Q3/Q4 독립 감사


## 2. 데이터·평가지표·연도 분할·모델 함수

In [2]:
def load_frame(group, split):
    path = PREPROCESSED_DIR / f"{split}_group{group}_preprocessed.csv"
    frame = pd.read_csv(path, low_memory=False)
    frame[TIME_COL] = pd.to_datetime(frame[TIME_COL], errors="raise")
    return frame.sort_values(TIME_COL).reset_index(drop=True)


def official_metrics(actual_kwh, prediction_kwh, capacity_kwh):
    actual = np.asarray(actual_kwh, dtype=float)
    pred = np.asarray(prediction_kwh, dtype=float)
    mask = np.isfinite(actual) & np.isfinite(pred) & (actual >= ACTIVE_CF * capacity_kwh)
    if not mask.any():
        return {"NMAE": np.nan, "FICR": np.nan, "Score": np.nan, "rows": 0,
                "within_6": np.nan, "within_8": np.nan}
    error_cf = np.abs(pred[mask] - actual[mask]) / capacity_kwh
    nmae = float(error_cf.mean())
    rate = np.where(error_cf <= 0.06, 4.0, np.where(error_cf <= 0.08, 3.0, 0.0))
    ficr = float(np.sum(rate * actual[mask]) / np.sum(4.0 * actual[mask]))
    return {"NMAE": nmae, "FICR": ficr, "Score": 0.5 * (1.0 - nmae + ficr),
            "rows": int(mask.sum()), "within_6": float((error_cf <= .06).mean()),
            "within_8": float((error_cf <= .08).mean())}


def split_metrics(actual_kwh, prediction_kwh, timestamps, capacity_kwh):
    t = pd.to_datetime(pd.Series(timestamps)).reset_index(drop=True)
    actual = np.asarray(actual_kwh, dtype=float)
    pred = np.asarray(prediction_kwh, dtype=float)
    masks = {
        "Q3": t.dt.month.between(7, 9).to_numpy(),
        "Q4": t.dt.month.between(10, 12).to_numpy(),
        "FULL": np.ones(len(t), dtype=bool),
    }
    return {name: official_metrics(actual[m], pred[m], capacity_kwh) for name, m in masks.items()}


def gain_table(actual_cf, candidate_cf, baseline_cf, timestamps, capacity):
    cand = split_metrics(actual_cf * capacity, candidate_cf * capacity, timestamps, capacity)
    base = split_metrics(actual_cf * capacity, baseline_cf * capacity, timestamps, capacity)
    gain = {q: cand[q]["Score"] - base[q]["Score"] for q in ("Q3", "Q4", "FULL")}
    return cand, base, gain


def passes_audit(gain):
    return bool(gain["Q3"] >= MIN_Q_GAIN and gain["Q4"] >= MIN_Q_GAIN
                and gain["FULL"] >= MIN_FULL_GAIN)


def masks_for_group(frame, group):
    t = frame[TIME_COL]
    if group in (1, 2):
        select_fit = t.dt.year.eq(2022).to_numpy()
        select_val = t.dt.year.eq(2023).to_numpy()
    else:
        select_fit = (t.dt.year.eq(2023) & t.dt.month.le(6)).to_numpy()
        select_val = (t.dt.year.eq(2023) & t.dt.month.ge(7)).to_numpy()
    audit_fit = (t.dt.year < 2024).to_numpy()
    audit_val = t.dt.year.eq(2024).to_numpy()
    final_fit = (t.dt.year <= 2024).to_numpy() & pd.to_numeric(frame[TARGET[group]], errors="coerce").notna().to_numpy()
    for name, mask in {"select_fit": select_fit, "select_val": select_val,
                       "audit_fit": audit_fit, "audit_val": audit_val}.items():
        if mask.sum() < 1000:
            raise RuntimeError(f"Group {group} {name} 행이 부족합니다: {mask.sum()}")
    if np.any(select_fit & select_val) or np.any(audit_fit & audit_val):
        raise AssertionError(f"Group {group}: 연도전진 학습/검증 구간이 겹칩니다.")
    if t.loc[select_fit].max() >= t.loc[select_val].min():
        raise AssertionError(f"Group {group}: 선택 구간이 시간 순서를 위반합니다.")
    if t.loc[audit_fit].max() >= t.loc[audit_val].min():
        raise AssertionError(f"Group {group}: 2024 감사 구간이 시간 순서를 위반합니다.")
    if np.any(select_val & ~audit_fit):
        raise AssertionError(f"Group {group}: 선택 검증 라벨이 감사 구간으로 누출됩니다.")
    return select_fit, select_val, audit_fit, audit_val, final_fit


def choose_features(frame, group, fit_mask):
    target = pd.to_numeric(frame[TARGET[group]], errors="coerce")
    lstm_col = f"group{group}_lstm_turbine_energy_kwh"
    if lstm_col not in frame.columns:
        raise KeyError(f"신규 LSTM 피처가 없습니다: {lstm_col}")
    blocked = ("kpx_group_", "target", "actual", "prediction", "outage", "scada",
               "turbine_energy", "10min_energy", "power_kw")
    ranked = []
    for col in frame.columns:
        if col in (TIME_COL, TARGET[group], lstm_col) or any(x in col.lower() for x in blocked):
            continue
        x = pd.to_numeric(frame[col], errors="coerce")
        valid = fit_mask & x.notna().to_numpy() & target.notna().to_numpy()
        if valid.sum() < 300 or float(x.loc[valid].std()) < 1e-10:
            continue
        corr = x.loc[valid].corr(target.loc[valid], method="spearman")
        ranked.append((abs(float(corr)) if np.isfinite(corr) else 0.0, col))
    columns = [lstm_col] + [c for _, c in sorted(ranked, reverse=True)[:MAX_FEATURES - 1]]
    if columns[0] != lstm_col:
        raise AssertionError("신규 LSTM 피처 강제 포함 실패")
    return columns


def numeric_matrix(frame, columns, medians=None, fit_mask=None):
    matrix = frame[columns].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    if medians is None:
        if fit_mask is None:
            raise ValueError("medians 또는 fit_mask가 필요합니다.")
        medians = matrix.loc[fit_mask].median().fillna(0.0)
    return matrix.fillna(medians).fillna(0.0).astype("float32"), medians


def candidate_specs(group):
    # Fast/Full은 계산 규모만 다르고 후보 구조는 반드시 동일합니다.
    specs = [
        ("direct_lgb_mae", "lgb", "mae", None, "all"),
        ("active_lgb_mae", "lgb", "mae", None, "active"),
        ("active_lgb_q58", "lgb", "quantile", 0.58, "active"),
        ("active_cat_q55", "cat", "quantile", 0.55, "active"),
        ("high_weight_lgb", "lgb", "mae", None, "high_weight"),
        ("active_lgb_q65", "lgb", "quantile", 0.65, "active"),
    ]
    return specs


def make_regressor(kind, loss, alpha, iterations):
    if kind == "lgb":
        params = dict(objective="regression_l1" if loss == "mae" else "quantile",
                      n_estimators=int(iterations), learning_rate=.025, num_leaves=39,
                      max_depth=8, min_child_samples=55, subsample=.85, subsample_freq=1,
                      colsample_bytree=.80, reg_alpha=.35, reg_lambda=7.0,
                      random_state=SEED, n_jobs=-1, verbosity=-1)
        if alpha is not None:
            params["alpha"] = alpha
        return lgb.LGBMRegressor(**params)
    loss_function = "MAE" if loss == "mae" else f"Quantile:alpha={alpha}"
    return CatBoostRegressor(loss_function=loss_function, eval_metric=loss_function,
                             iterations=int(iterations), learning_rate=.03, depth=8,
                             l2_leaf_reg=7.0, random_seed=SEED, verbose=False,
                             allow_writing_files=False, thread_count=-1)


def fit_bundle(frame, group, fit_mask, pred_mask, columns, fixed_iterations=None, final=False):
    X, medians = numeric_matrix(frame, columns, fit_mask=fit_mask)
    y = pd.to_numeric(frame[TARGET[group]], errors="coerce").to_numpy(float) / CAPACITY[group]
    active = y >= ACTIVE_CF
    predictions, iterations, models = {}, {}, {}
    for name, kind, loss, alpha, region in candidate_specs(group):
        model_fit = fit_mask.copy()
        weights = np.ones(model_fit.sum(), dtype=float)
        if region in ("active", "high_weight"):
            model_fit &= active
            weights = np.ones(model_fit.sum(), dtype=float)
        if region == "high_weight":
            weights *= 1.0 + 2.0 * np.square(y[model_fit])
        limit = LGB_TREES if kind == "lgb" else CAT_TREES
        n_iter = int(fixed_iterations.get(name, limit) if fixed_iterations else limit)
        if final:
            n_iter = max(100, int(np.ceil(n_iter * FINAL_ITERATION_MULTIPLIER)))
        model = make_regressor(kind, loss, alpha, n_iter)
        if fixed_iterations is None:
            eval_mask = pred_mask & (active if region in ("active", "high_weight") else True)
            if kind == "lgb":
                model.fit(X.loc[model_fit], y[model_fit], sample_weight=weights,
                          eval_set=[(X.loc[eval_mask], y[eval_mask])],
                          callbacks=[lgb.early_stopping(120, verbose=False)])
                best_iter = int(model.best_iteration_ or n_iter)
            else:
                model.fit(X.loc[model_fit], y[model_fit], sample_weight=weights,
                          eval_set=(X.loc[eval_mask], y[eval_mask]),
                          early_stopping_rounds=120, verbose=False)
                best_iter = max(1, int(model.get_best_iteration() + 1))
        else:
            model.fit(X.loc[model_fit], y[model_fit], sample_weight=weights)
            best_iter = n_iter
        predictions[name] = np.clip(model.predict(X.loc[pred_mask]), 0.0, PRED_MAX_CF)
        iterations[name] = best_iter
        models[name] = model

    classifier = lgb.LGBMClassifier(objective="binary", n_estimators=ACTIVE_CLASSIFIER_TREES,
                                    learning_rate=.025, num_leaves=31, max_depth=7,
                                    min_child_samples=60, reg_alpha=.3, reg_lambda=6.0,
                                    random_state=SEED, n_jobs=-1, verbosity=-1)
    classifier.fit(X.loc[fit_mask], active[fit_mask].astype(np.int8))
    active_probability = classifier.predict_proba(X.loc[pred_mask])[:, 1]

    high_classifier = None
    high_probability = None
    high_expert = None
    high_expert_prediction = None
    if group == 3:
        high_target = y >= .80
        high_classifier = lgb.LGBMClassifier(objective="binary", n_estimators=HIGH_CLASSIFIER_TREES,
                                             learning_rate=.025, num_leaves=25, max_depth=6,
                                             min_child_samples=50, reg_alpha=.4, reg_lambda=7.0,
                                             random_state=SEED, n_jobs=-1, verbosity=-1)
        high_classifier.fit(X.loc[fit_mask], high_target[fit_mask].astype(np.int8))
        high_probability = high_classifier.predict_proba(X.loc[pred_mask])[:, 1]
        high_fit = fit_mask & (y >= .70)
        high_n_iter = int(fixed_iterations.get("g3_high_expert", LGB_TREES) if fixed_iterations else LGB_TREES)
        if final:
            high_n_iter = max(100, int(np.ceil(high_n_iter * FINAL_ITERATION_MULTIPLIER)))
        high_expert = make_regressor("lgb", "quantile", .65, high_n_iter)
        if fixed_iterations is None:
            val_high = pred_mask & (y >= .70)
            high_expert.fit(X.loc[high_fit], y[high_fit], eval_set=[(X.loc[val_high], y[val_high])],
                            callbacks=[lgb.early_stopping(120, verbose=False)])
            iterations["g3_high_expert"] = int(high_expert.best_iteration_ or LGB_TREES)
        else:
            high_expert.fit(X.loc[high_fit], y[high_fit])
            iterations["g3_high_expert"] = int(fixed_iterations.get("g3_high_expert", LGB_TREES))
        high_expert_prediction = np.clip(high_expert.predict(X.loc[pred_mask]), 0.0, PRED_MAX_CF)
    return {"pred": predictions, "iterations": iterations, "active_probability": active_probability,
            "high_probability": high_probability, "high_expert": high_expert_prediction,
            "models": models, "active_classifier": classifier, "high_classifier": high_classifier,
            "high_expert_model": high_expert, "medians": medians, "columns": columns}

## 3. 앙상블·하한 보정·Gaussian MoE·고출력 Gate·그룹 후처리

In [3]:
def weight_candidates(count, trials):
    eye = np.eye(count)
    yield from eye
    yield np.ones(count) / count
    rng = np.random.default_rng(SEED)
    for row in rng.dirichlet(np.ones(count) * .7, size=trials):
        yield row


def apply_anchor(prediction, probability, threshold, low=.16, high=.18):
    pred = np.asarray(prediction, float).copy()
    prob = np.asarray(probability, float)
    inactive = prob < threshold
    relative = np.clip(prob / max(threshold, 1e-6), 0.0, 1.0)
    anchor = low + (high - low) * relative
    pred[inactive] = anchor[inactive]
    return np.clip(pred, 0.0, PRED_MAX_CF)


def apply_blend(bundle, recipe):
    matrix = np.column_stack([bundle["pred"][name] for name in recipe["names"]])
    prediction = np.clip(matrix @ np.asarray(recipe["weights"], float), 0.0, PRED_MAX_CF)
    return apply_anchor(prediction, bundle["active_probability"], recipe["inactive_threshold"])


def select_blend_recipe(frame, group, pred_mask, bundle):
    y = frame.loc[pred_mask, TARGET[group]].to_numpy(float) / CAPACITY[group]
    times = frame.loc[pred_mask, TIME_COL].reset_index(drop=True)
    names = list(bundle["pred"])
    baseline = bundle["pred"]["direct_lgb_mae"]
    candidates = []
    for weights in weight_candidates(len(names), BLEND_TRIALS):
        raw = np.clip(np.column_stack([bundle["pred"][n] for n in names]) @ weights, 0, PRED_MAX_CF)
        for threshold in (.45, .50, .55, .60, .65):
            pred = apply_anchor(raw, bundle["active_probability"], threshold)
            metrics, _, gain = gain_table(y, pred, baseline, times, CAPACITY[group])
            # 선택연도에서도 Q3와 Q4를 동시에 보호합니다.
            if gain["Q3"] < 0 or gain["Q4"] < 0:
                continue
            candidates.append((min(gain["Q3"], gain["Q4"]), metrics["FULL"]["Score"],
                               weights.copy(), threshold, gain))
    if not candidates:
        weights = np.zeros(len(names)); weights[names.index("direct_lgb_mae")] = 1.0
        return {"names": names, "weights": weights.tolist(), "inactive_threshold": .0,
                "selection_gain": {"Q3": 0., "Q4": 0., "FULL": 0.}, "fallback": True}
    best = max(candidates, key=lambda x: (x[0], x[1]))
    return {"names": names, "weights": best[2].tolist(), "inactive_threshold": float(best[3]),
            "selection_gain": best[4], "fallback": False}


def fit_g2_experts(frame, fit_mask, pred_mask, columns, fixed_iterations=None, final=False):
    X, _ = numeric_matrix(frame, columns, fit_mask=fit_mask)
    y = frame[TARGET[2]].to_numpy(float) / CAPACITY[2]
    centers = np.array([.20, .45, .70, .90])
    sigmas = np.array([.10, .14, .13, .10])
    outputs, models = [], []
    n_iter = int(fixed_iterations or G2_EXPERT_TREES)
    if final:
        n_iter = int(np.ceil(n_iter * FINAL_ITERATION_MULTIPLIER))
    for center, sigma in zip(centers, sigmas):
        weights = .10 + .90 * np.exp(-.5 * ((y[fit_mask] - center) / sigma) ** 2)
        model = lgb.LGBMRegressor(objective="regression_l1", n_estimators=n_iter,
                                  learning_rate=.025, num_leaves=31, max_depth=7,
                                  min_child_samples=50, colsample_bytree=.82,
                                  reg_alpha=.35, reg_lambda=7.0, random_state=SEED,
                                  n_jobs=-1, verbosity=-1)
        model.fit(X.loc[fit_mask], y[fit_mask], sample_weight=weights)
        outputs.append(np.clip(model.predict(X.loc[pred_mask]), 0, PRED_MAX_CF)); models.append(model)
    return np.column_stack(outputs), models, centers, sigmas, n_iter


def gaussian_moe(base, experts, params):
    base = np.asarray(base, float); experts = np.asarray(experts, float)
    centers = np.asarray(params["centers"], float); sigmas = np.asarray(params["sigmas"], float)
    amplitudes = np.asarray(params["amplitudes"], float); scales = np.asarray(params["scales"], float)
    weights = amplitudes[None, :] * np.exp(-.5 * ((base[:, None] - centers[None, :]) / sigmas[None, :]) ** 2)
    weights /= np.maximum(weights.sum(axis=1, keepdims=True), 1e-12)
    delta = np.sum(weights * scales[None, :] * (experts - base[:, None]), axis=1)
    disagreement = np.sqrt(np.sum(weights * (experts - np.sum(weights * experts, axis=1)[:, None]) ** 2, axis=1))
    strength = params["alpha"] * np.exp(-params["damping"] * disagreement)
    return np.clip(base + np.clip(strength * delta, -params["cap"], params["cap"]), 0, PRED_MAX_CF)


def select_g2_moe(frame, pred_mask, base, experts, centers, sigmas):
    y = frame.loc[pred_mask, TARGET[2]].to_numpy(float) / CAPACITY[2]
    times = frame.loc[pred_mask, TIME_COL].reset_index(drop=True)
    rng = np.random.default_rng(SEED)
    params_list = [{"centers": centers.tolist(), "sigmas": sigmas.tolist(),
                    "amplitudes": [1.] * 4, "scales": [1.] * 4,
                    "alpha": .35, "damping": 3., "cap": .06}]
    for _ in range(MOE_TRIALS):
        params_list.append({"centers": np.clip(centers + rng.normal(0, .025, 4), .12, .96).tolist(),
                            "sigmas": np.clip(sigmas * rng.uniform(.75, 1.35, 4), .06, .22).tolist(),
                            "amplitudes": rng.uniform(.55, 1.35, 4).tolist(),
                            "scales": rng.uniform(.20, 1.25, 4).tolist(),
                            "alpha": float(rng.uniform(.10, .70)),
                            "damping": float(rng.uniform(1.5, 5.0)),
                            "cap": float(rng.uniform(.025, .08))})
    accepted = []
    for params in params_list:
        pred = gaussian_moe(base, experts, params)
        metrics, _, gain = gain_table(y, pred, base, times, CAPACITY[2])
        if gain["Q3"] >= 0 and gain["Q4"] >= 0:
            accepted.append((min(gain["Q3"], gain["Q4"]), metrics["FULL"]["Score"], params, gain))
    if not accepted:
        return None
    best = max(accepted, key=lambda x: (x[0], x[1]))
    best[2]["selection_gain"] = best[3]
    return best[2]


def apply_g3_gate(base, expert, probability, params):
    if params is None:
        return np.asarray(base, float)
    trust = np.clip((np.asarray(probability) - params["threshold"]) /
                    max(1e-6, 1.0 - params["threshold"]), 0, 1) ** params["eta"]
    gap = np.maximum(np.asarray(expert) - np.asarray(base), 0.0)
    return np.clip(np.asarray(base) + np.minimum(params["gamma"] * trust * gap, params["cap"]), 0, PRED_MAX_CF)


def select_g3_gate(frame, pred_mask, base, expert, probability):
    y = frame.loc[pred_mask, TARGET[3]].to_numpy(float) / CAPACITY[3]
    times = frame.loc[pred_mask, TIME_COL].reset_index(drop=True)
    accepted = []
    for threshold in (.20, .25, .30, .35, .40, .50):
        for gamma in (.15, .25, .35, .50):
            for eta in (1.0, 1.5, 2.0):
                params = {"threshold": threshold, "gamma": gamma, "eta": eta, "cap": .10}
                pred = apply_g3_gate(base, expert, probability, params)
                metrics, _, gain = gain_table(y, pred, base, times, CAPACITY[3])
                if gain["Q3"] >= 0 and gain["Q4"] >= 0:
                    accepted.append((min(gain["Q3"], gain["Q4"]), metrics["FULL"]["Score"], params, gain))
    if not accepted:
        return None
    best = max(accepted, key=lambda x: (x[0], x[1]))
    best[2]["selection_gain"] = best[3]
    return best[2]


def select_group1_postprocess(frame, pred_mask, group1_cf, group2_cf):
    y = frame.loc[pred_mask, TARGET[1]].to_numpy(float) / CAPACITY[1]
    times = frame.loc[pred_mask, TIME_COL].reset_index(drop=True)
    accepted = []
    for reference_strength in (0., .05, .10, .15, .20):
        for scale in (.98, .99, 1., 1.01, 1.02):
            for offset in (-.01, -.005, 0., .005, .01):
                pred = np.clip((group1_cf + reference_strength * (group2_cf - group1_cf)) * scale + offset,
                               0, PRED_MAX_CF)
                metrics, _, gain = gain_table(y, pred, group1_cf, times, CAPACITY[1])
                if gain["Q3"] >= 0 and gain["Q4"] >= 0:
                    accepted.append((min(gain["Q3"], gain["Q4"]), metrics["FULL"]["Score"],
                                     {"reference_strength": reference_strength, "scale": scale,
                                      "offset": offset, "selection_gain": gain}))
    return max(accepted, key=lambda x: (x[0], x[1]))[2] if accepted else None


def apply_group1_postprocess(group1_cf, group2_cf, params):
    if params is None:
        return np.asarray(group1_cf, float)
    return np.clip((np.asarray(group1_cf) + params["reference_strength"] *
                    (np.asarray(group2_cf) - np.asarray(group1_cf))) * params["scale"] + params["offset"],
                   0, PRED_MAX_CF)


def align_by_time(source_times, source_values, target_times):
    """서로 한 행 차이가 날 수 있는 그룹별 시각을 안전하게 맞춥니다."""
    source = pd.Series(np.asarray(source_values, float), index=pd.to_datetime(source_times))
    source = source.groupby(level=0).mean()
    target_index = pd.DatetimeIndex(pd.to_datetime(pd.Series(target_times)))
    # 그룹별 정지/품질 처리로 드물게 특정 시각이 한 그룹에만 없습니다.
    # 라벨을 쓰지 않고 모델 예측 시계열만 시간 보간하여 참조 피처를 맞춥니다.
    union = source.index.union(target_index).sort_values()
    source = source.reindex(union).interpolate(method="time", limit_direction="both")
    aligned = pd.Series(target_index.map(source), index=np.arange(len(target_index)))
    if aligned.isna().any():
        missing = pd.to_datetime(pd.Series(target_times))[aligned.isna()].head(5).astype(str).tolist()
        raise RuntimeError(f"그룹 참조 후처리 시간 정렬 실패: {missing}")
    return aligned.to_numpy(float)

## 4. 연도전진 선택 → 2024 감사 → 최종 재학습

In [4]:
started = time.perf_counter()
train_frames = {g: load_frame(g, "train") for g in GROUPS}
test_frames = {g: load_frame(g, "test") for g in GROUPS}
split_masks = {g: masks_for_group(train_frames[g], g) for g in GROUPS}
split_summary = {}
for g in GROUPS:
    sf, sv, af, av, _ = split_masks[g]
    t = train_frames[g][TIME_COL]
    split_summary[g] = {
        "selection_fit": [str(t.loc[sf].min()), str(t.loc[sf].max()), int(sf.sum())],
        "selection_validation": [str(t.loc[sv].min()), str(t.loc[sv].max()), int(sv.sum())],
        "audit_fit": [str(t.loc[af].min()), str(t.loc[af].max()), int(af.sum())],
        "audit_2024": [str(t.loc[av].min()), str(t.loc[av].max()), int(av.sum())],
    }
    print(f"Group {g} 연도전진: select {split_summary[g]['selection_fit']} -> "
          f"{split_summary[g]['selection_validation']}; audit -> {split_summary[g]['audit_2024']}")
features = {g: choose_features(train_frames[g], g, split_masks[g][0]) for g in GROUPS}

print("\n[1/5] 선택연도 모델 학습: 설정은 여기서만 탐색합니다.", flush=True)
selection_bundles = {}
for g in GROUPS:
    sf, sv, _, _, _ = split_masks[g]
    selection_bundles[g] = fit_bundle(train_frames[g], g, sf, sv, features[g])
    print(f"  Group {g}: 후보 완료, LSTM={features[g][0]}", flush=True)

recipes = {g: select_blend_recipe(train_frames[g], g, split_masks[g][1], selection_bundles[g]) for g in GROUPS}
selection_base = {g: apply_blend(selection_bundles[g], recipes[g]) for g in GROUPS}

# Group 2 Gaussian MoE: 2022 학습→2023 선택에서만 탐색합니다.
g2_experts_sel, _, g2_centers, g2_sigmas, g2_expert_iterations = fit_g2_experts(
    train_frames[2], split_masks[2][0], split_masks[2][1], features[2])
g2_moe_params = select_g2_moe(train_frames[2], split_masks[2][1], selection_base[2],
                              g2_experts_sel, g2_centers, g2_sigmas)
if g2_moe_params is not None:
    selection_base[2] = gaussian_moe(selection_base[2], g2_experts_sel, g2_moe_params)

# Group 3 제한형 고출력 Gate: 2023 상반기→하반기에서만 탐색합니다.
g3_gate_params = select_g3_gate(train_frames[3], split_masks[3][1], selection_base[3],
                                selection_bundles[3]["high_expert"],
                                selection_bundles[3]["high_probability"])
selection_base[3] = apply_g3_gate(selection_base[3], selection_bundles[3]["high_expert"],
                                  selection_bundles[3]["high_probability"], g3_gate_params)

# Group 1은 같은 시각 Group 2 예측을 약하게 참조하는 후처리를 선택연도에서만 탐색합니다.
g1_post_params = select_group1_postprocess(train_frames[1], split_masks[1][1],
                                            selection_base[1],
                                            align_by_time(train_frames[2].loc[split_masks[2][1], TIME_COL],
                                                          selection_base[2],
                                                          train_frames[1].loc[split_masks[1][1], TIME_COL]))
selection_g2_for_g1 = align_by_time(train_frames[2].loc[split_masks[2][1], TIME_COL], selection_base[2],
                                    train_frames[1].loc[split_masks[1][1], TIME_COL])
selection_base[1] = apply_group1_postprocess(selection_base[1], selection_g2_for_g1, g1_post_params)

print("\n[2/5] 설정 고정 완료. 2024 라벨을 보지 않고 재학습·예측합니다.", flush=True)
audit_bundles = {}
for g in GROUPS:
    _, _, af, av, _ = split_masks[g]
    audit_bundles[g] = fit_bundle(train_frames[g], g, af, av, features[g],
                                  fixed_iterations=selection_bundles[g]["iterations"])
    print(f"  Group {g}: 2024 감사 예측 완료", flush=True)

audit_base = {g: apply_blend(audit_bundles[g], recipes[g]) for g in GROUPS}
g2_experts_audit, _, _, _, _ = fit_g2_experts(train_frames[2], split_masks[2][2], split_masks[2][3],
                                               features[2], fixed_iterations=g2_expert_iterations)
if g2_moe_params is not None:
    audit_base[2] = gaussian_moe(audit_base[2], g2_experts_audit, g2_moe_params)
audit_base[3] = apply_g3_gate(audit_base[3], audit_bundles[3]["high_expert"],
                              audit_bundles[3]["high_probability"], g3_gate_params)

# Group 2가 2024 감사를 통과하지 못하면 Group 1 후처리도 실제 최종값인
# Group 2 직접 LGB를 참조해야 선택/제출 간 구조가 달라지지 않습니다.
g2_av = split_masks[2][3]
g2_y = train_frames[2].loc[g2_av, TARGET[2]].to_numpy(float) / CAPACITY[2]
g2_t = train_frames[2].loc[g2_av, TIME_COL].reset_index(drop=True)
g2_baseline = audit_bundles[2]["pred"]["direct_lgb_mae"]
_, _, g2_pre_gain = gain_table(g2_y, audit_base[2], g2_baseline, g2_t, CAPACITY[2])
g2_reference = audit_base[2] if passes_audit(g2_pre_gain) else g2_baseline
g2_reference_aligned = align_by_time(train_frames[2].loc[g2_av, TIME_COL], g2_reference,
                                     train_frames[1].loc[split_masks[1][3], TIME_COL])
audit_base[1] = apply_group1_postprocess(audit_base[1], g2_reference_aligned, g1_post_params)

print("\n[3/5] 2024 Q3·Q4 독립 감사: 하나라도 기준 미달이면 직접 LGB로 복귀합니다.", flush=True)
approved = {}
validation_rows = []
validation_predictions = []
for g in GROUPS:
    av = split_masks[g][3]
    y = train_frames[g].loc[av, TARGET[g]].to_numpy(float) / CAPACITY[g]
    t = train_frames[g].loc[av, TIME_COL].reset_index(drop=True)
    baseline = audit_bundles[g]["pred"]["direct_lgb_mae"]
    candidate = audit_base[g]
    cand_metrics, base_metrics, gain = gain_table(y, candidate, baseline, t, CAPACITY[g])
    approved[g] = passes_audit(gain)
    final_cf = candidate if approved[g] else baseline
    final_metrics = split_metrics(y * CAPACITY[g], final_cf * CAPACITY[g], t, CAPACITY[g])
    print(f"  Group {g}: gain Q3={gain['Q3']:+.6f}, Q4={gain['Q4']:+.6f}, FULL={gain['FULL']:+.6f} "
          f"→ {'승인' if approved[g] else 'LGB MAE 복귀'}", flush=True)
    for q in ("Q3", "Q4", "FULL"):
        validation_rows.append({"group": g, "split": "2024_AUDIT", "quarter": q,
                                "approved": approved[g], **final_metrics[q],
                                "gain_vs_direct_lgb": gain[q]})
    probability = audit_bundles[g]["active_probability"]
    validation_predictions.append(pd.DataFrame({
        "group": g, TIME_COL: t, "actual_kwh": y * CAPACITY[g],
        "prediction_kwh": final_cf * CAPACITY[g],
        "direct_lgb_kwh": baseline * CAPACITY[g],
        "active_probability": probability,
        "classifier_inactive": probability < recipes[g]["inactive_threshold"],
        "approved_year_forward": approved[g],
    }))

metrics_frame = pd.DataFrame(validation_rows)
pred_frame = pd.concat(validation_predictions, ignore_index=True)
metrics_frame.to_csv(OUTPUT_DIR / "year_forward_validation_metrics.csv", index=False, encoding="utf-8-sig")
pred_frame.to_csv(OUTPUT_DIR / "2024_validation_predictions.csv", index=False, encoding="utf-8-sig")

scale_config = {"max_features": MAX_FEATURES, "lgb_trees": LGB_TREES, "cat_trees": CAT_TREES,
                "blend_trials": BLEND_TRIALS, "moe_trials": MOE_TRIALS,
                "active_classifier_trees": ACTIVE_CLASSIFIER_TREES,
                "high_classifier_trees": HIGH_CLASSIFIER_TREES, "g2_expert_trees": G2_EXPERT_TREES}
frozen = {"mode": MODE, "architecture": "model2_year_forward_v3_shared",
          "candidate_names": [spec[0] for spec in candidate_specs(1)], "scale_config": scale_config,
          "validation": "G1/G2 2022->2023, G3 2023H1->H2; frozen audit 2024",
          "split_summary": split_summary,
          "minimum_gain": {"Q3": MIN_Q_GAIN, "Q4": MIN_Q_GAIN, "FULL": MIN_FULL_GAIN},
          "approved": approved, "blend_recipes": recipes, "g2_gaussian_moe": g2_moe_params,
          "g3_high_output_gate": g3_gate_params, "g1_postprocess": g1_post_params,
          "features": features}
(OUTPUT_DIR / "frozen_year_forward_config.json").write_text(
    json.dumps(frozen, ensure_ascii=False, indent=2), encoding="utf-8")

print("\n[4/5] 승인된 구조만 2024까지 포함해 최종 재학습합니다.", flush=True)
test_cf = {}
for g in GROUPS:
    frame = pd.concat([train_frames[g], test_frames[g]], ignore_index=True, sort=False)
    final_fit = np.zeros(len(frame), dtype=bool)
    final_fit[:len(train_frames[g])] = split_masks[g][4]
    test_mask = np.zeros(len(frame), dtype=bool); test_mask[len(train_frames[g]):] = True
    bundle = fit_bundle(frame, g, final_fit, test_mask, features[g],
                        fixed_iterations=selection_bundles[g]["iterations"], final=True)
    candidate = apply_blend(bundle, recipes[g])
    if g == 2 and g2_moe_params is not None:
        experts, _, _, _, _ = fit_g2_experts(frame, final_fit, test_mask, features[g],
                                              fixed_iterations=g2_expert_iterations, final=True)
        candidate = gaussian_moe(candidate, experts, g2_moe_params)
    if g == 3:
        candidate = apply_g3_gate(candidate, bundle["high_expert"], bundle["high_probability"], g3_gate_params)
    baseline = bundle["pred"]["direct_lgb_mae"]
    test_cf[g] = candidate if approved[g] else baseline
    joblib.dump({"models": bundle["models"], "active_classifier": bundle["active_classifier"],
                 "high_classifier": bundle["high_classifier"], "high_expert": bundle["high_expert_model"]},
                OUTPUT_DIR / "checkpoints" / f"group{g}_final_models.joblib")

if approved[1]:
    aligned_test_g2 = align_by_time(test_frames[2][TIME_COL], test_cf[2], test_frames[1][TIME_COL])
    test_cf[1] = apply_group1_postprocess(test_cf[1], aligned_test_g2, g1_post_params)

sample = pd.read_csv(ROOT / "wind_data" / "sample_submission.csv")
sample[TIME_COL] = pd.to_datetime(sample[TIME_COL])
for g in GROUPS:
    test_times = pd.to_datetime(test_frames[g][TIME_COL])
    series = pd.Series(test_cf[g] * CAPACITY[g], index=test_times).groupby(level=0).mean()
    sample[TARGET[g]] = sample[TIME_COL].map(series)
    if sample[TARGET[g]].isna().any():
        raise RuntimeError(f"Group {g}: sample_submission 시간과 test 시간이 일치하지 않습니다.")
sample.to_csv(OUTPUT_DIR / "submission.csv", index=False, encoding="utf-8-sig",
              date_format="%Y-%m-%d %H:%M:%S")

print("\n[5/5] 완료")
print("제출:", OUTPUT_DIR / "submission.csv")
print("검증:", OUTPUT_DIR / "year_forward_validation_metrics.csv")
print(f"실행시간: {(time.perf_counter() - started) / 60:.1f}분")
display(metrics_frame)

Group 1 연도전진: select ['2022-01-01 01:00:00', '2022-12-31 23:00:00', 8664] -> ['2023-01-01 00:00:00', '2023-12-31 23:00:00', 8757]; audit -> ['2024-01-01 00:00:00', '2024-12-31 23:00:00', 8778]
Group 2 연도전진: select ['2022-01-01 01:00:00', '2022-12-31 23:00:00', 8664] -> ['2023-01-01 00:00:00', '2023-12-31 23:00:00', 8758]; audit -> ['2024-01-01 00:00:00', '2024-12-31 23:00:00', 8778]
Group 3 연도전진: select ['2023-01-01 01:00:00', '2023-06-30 23:00:00', 4343] -> ['2023-07-01 00:00:00', '2023-12-31 23:00:00', 4416]; audit -> ['2024-01-01 00:00:00', '2024-12-31 23:00:00', 8778]

[1/5] 선택연도 모델 학습: 설정은 여기서만 탐색합니다.
  Group 1: 후보 완료, LSTM=group1_lstm_turbine_energy_kwh
  Group 2: 후보 완료, LSTM=group2_lstm_turbine_energy_kwh
  Group 3: 후보 완료, LSTM=group3_lstm_turbine_energy_kwh

[2/5] 설정 고정 완료. 2024 라벨을 보지 않고 재학습·예측합니다.
  Group 1: 2024 감사 예측 완료
  Group 2: 2024 감사 예측 완료
  Group 3: 2024 감사 예측 완료

[3/5] 2024 Q3·Q4 독립 감사: 하나라도 기준 미달이면 직접 LGB로 복귀합니다.
  Group 1: gain Q3=+0.051639, Q4=+0.074865, FULL=+0.0

,group,split,quarter,approved,NMAE,FICR,Score,rows,within_6,within_8,gain_vs_direct_lgb
0,1,2024_AUDIT,Q3,True,0.114955,0.423822,0.654433,1072,0.372201,0.483209,0.051639
1,1,2024_AUDIT,Q4,True,0.104079,0.495775,0.695848,1371,0.419402,0.519329,0.074865
2,1,2024_AUDIT,FULL,True,0.116611,0.448464,0.665927,4989,0.370014,0.468431,0.046839
3,2,2024_AUDIT,Q3,True,0.111368,0.469757,0.679194,1068,0.399813,0.492509,0.016366
4,2,2024_AUDIT,Q4,True,0.109560,0.478419,0.684430,1329,0.411588,0.496614,0.048192
5,2,2024_AUDIT,FULL,True,0.124539,0.466410,0.670936,4976,0.370780,0.463424,0.020216
6,3,2024_AUDIT,Q3,True,0.129217,0.310558,0.590670,963,0.329180,0.414330,0.043230
7,3,2024_AUDIT,Q4,True,0.122842,0.374988,0.626073,1196,0.329431,0.434783,0.033993
8,3,2024_AUDIT,FULL,True,0.130809,0.314332,0.591762,4566,0.316251,0.406483,0.035059
